# Titanic - Predict Survival

**Goal:** Predict which passengers survived the Titanic
**Algorithm:** Random Forest + Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
%matplotlib inline

In [ ]:
np.random.seed(42)
n = 891

pclass = np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55])
sex    = np.random.choice(['male', 'female'], n, p=[0.65, 0.35])
age    = np.clip(np.random.normal(30, 14, n).astype(int), 1, 80)
sibsp  = np.random.poisson(0.5, n)
parch  = np.random.poisson(0.3, n)
fare   = np.random.gamma(5, 10, n).round(2)
embarked = np.random.choice(['S', 'C', 'Q'], n, p=[0.7, 0.2, 0.1])

# Simulate survival based on rules
survived = ((sex == 'female') * 0.5 +
            (pclass == 1) * 0.2 +
            (age < 15) * 0.3 +
            np.random.random(n) * 0.3)
survived = (survived > 0.5).astype(int)

df = pd.DataFrame({
    'Pclass': pclass,
    'Sex': sex,
    'Age': age,
    'SibSp': sibsp,
    'Parch': parch,
    'Fare': fare,
    'Embarked': embarked,
    'Survived': survived
})
print ('Shape: %s' % (df.shape,))

<hr>## 1. Exploratory Data Analysis

In [ ]:
print ('Survival distribution:\n%s' % df['Survived'].value_counts())
print ('\nFirst 5 rows:\n%s' % df.head())
print ('\nMissing values:\n%s' % df.isnull().sum()[df.isnull().sum() > 0])

<hr>## 2. Feature Engineering

In [ ]:
# Create new features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Age fill with median (if missing)
null_idx = np.random.choice(df.index, 50, replace=False)
df.loc[null_idx, 'Age'] = np.nan
df['Age'] = df['Age'].fillna(df['Age'].median())

# Encode categoricals
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df['Embarked'] = df['Embarked'].fillna(0)

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
            'Embarked', 'FamilySize', 'IsAlone']

print ('Features: %s' % features)
print ('\nFirst 5 rows after engineering:\n%s' % df[features].head())

<hr>## 3. Train Model

In [ ]:
X = df[features]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print ('Train: %s, Test: %s' % (X_train.shape[0], X_test.shape[0]))

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print ('Model: %s' % model)

<hr>## 4. Evaluate Performance

In [ ]:
y_pred = model.predict(X_test)
print ('Accuracy: %.4f' % accuracy_score(y_test, y_pred))
print ('\nClassification Report:\n%s' % classification_report(y_test, y_pred))

<hr>## 5. Feature Importance

In [ ]:
importances = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print ('Feature Importance:\n%s' % importances.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(importances['feature'], importances['importance'])
plt.xlabel('Importance')
plt.title('Titanic Survival - Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

<hr>## 6. Predict on New Passenger

In [ ]:
# Predict for a sample passenger
sample = pd.DataFrame([{
    'Pclass': 1, 'Sex': 1, 'Age': 28,
    'SibSp': 0, 'Parch': 0, 'Fare': 100,
    'Embarked': 0, 'FamilySize': 1, 'IsAlone': 1
}])

pred = model.predict(sample)[0]
prob = model.predict_proba(sample)[0]
print ('Passenger: 1st class, Female, 28yrs, traveling alone')
print ('Survived: %s (probability: %.2f%%)' % ('Yes' if pred else 'No', prob[1]*100))